# Optimal control and parameter estimation on GPUs

Tutorial 1 simulated dynamics forward. This tutorial solves the **optimal
control problem** associated with the same dynamics: choose the control so
the system arrives where we want it, at least cost, as a nonlinear program.

[ExaModels.jl](https://github.com/exanauts/ExaModels.jl) turns the model
into GPU-ready evaluation and derivatives;
[MadNLP.jl](https://github.com/MadNLP/MadNLP.jl) solves it. Every model is
written out in full: formulating is what this tutorial teaches.

## Setup

On the CPU we solve with **Ipopt**, through `NLPModelsIpopt`, because Ipopt
is the reference interior-point code in this field and is the baseline a
CPU number should be measured against. On the GPU we use **MadNLP**. Both
consume the same ExaModels model, and both report `objective`, `iter` and a
status. The device comparison later is therefore a comparison of two
solvers as well as two devices.

In [ ]:
using CUDA, CUDSS, ExaModels, MadNLP, MadNLPGPU, NLPModelsIpopt
using Plots

# Vector output: plots stay sharp at any zoom, and the notebook stays
# readable when rendered to HTML.
gr(fmt = :svg)

## A warm-up solve

**Run the next cell now, then keep listening.** The first interior-point
solve on the GPU in a fresh session compiles the entire solver stack, which
takes roughly two to three minutes. Tutorial 1 showed the small version of
this effect. It is paid **once per session**, and the second solve takes a
fraction of a second. The problem below is deliberately trivial. Its only
job is to make the compiler do all of its work now, rather than in the
middle of an exercise. Each *new* model structure later adds a few seconds
of compilation on its first solve. That part is unavoidable, but it is
seconds rather than minutes.

In [ ]:
function warmup(backend)
    wc = ExaCore(; backend = backend)
    @add_var(wc, x, 100; start = 0.5)
    @add_obj(wc, (x[i] - 1.0)^2 for i = 1:100)
    @add_con(wc, x[i] + x[i+1] for i = 1:99; lcon = -10.0, ucon = 10.0)
    m = ExaModel(wc)
    return backend === nothing ? ipopt(m; max_iter = 600, print_level = 0).status :
                                 madnlp(m; max_iter = 600).status
end
(cpu = warmup(nothing), gpu = warmup(CUDABackend()))

Those six lines are the whole ExaModels API: variables, objective,
constraints, model.

## A first optimal control problem

The pendulum from tutorial 1, now with a motor at the pivot. Which torque
`u(t)` swings it from hanging down (θ = 0) to upright (θ = π) with least
effort? The torque is capped well below gravity, so the answer has to
*pump* the swing.

![Pendulum swing-up: the torque is too weak to lift the mass directly](figs/swingup.svg)

**Direct transcription** turns the optimal control problem into an NLP. Lay
a uniform grid of `N` intervals over the horizon, make the states `θ, ω`
and the control `u` at each grid point decision variables, and impose the
dynamics as constraints linking neighbours, here with the trapezoidal rule:

$$
\begin{aligned}
\min_{\theta, \omega, u}\quad & \sum_{i=0}^{N} u_i^2\, \Delta t \\
\text{s.t.}\quad
  & \theta_i - \theta_{i-1} - \tfrac{\Delta t}{2}(\omega_i + \omega_{i-1}) = 0,
  && i = 1, \dots, N, \\
  & \omega_i - \omega_{i-1} - \tfrac{\Delta t}{2}\big(f(\theta_i,\omega_i,u_i)
    + f(\theta_{i-1},\omega_{i-1},u_{i-1})\big) = 0,
  && i = 1, \dots, N, \\
  & -u_{\max} \le u_i \le u_{\max}, && i = 0, \dots, N, \\
  & \theta_0 = 0,\quad \omega_0 = 0,\quad \theta_N = \pi,\quad \omega_N = 0, \\[2pt]
  & \text{where } f(\theta, \omega, u) = u - 9.81 \sin\theta - 0.1\, \omega .
\end{aligned}
$$

Each line is one pattern repeated over its index, and each becomes a
single ExaModels generator: the two
dynamics families are `N` copies of one expression, the bound is `N+1`
copies of another, and only the four boundary conditions are one-offs.

What comes out is a **structured NLP**: large, sparse, block-banded, and
above all repetitive, which is exactly what ExaModels is built to exploit.
Every constraint is written as a *generator* over the time grid, the SIMD pattern from the lecture: one
expression, `N` instances. ExaModels compiles that into a single GPU kernel,
and differentiates it on the GPU as well. We wrap the model in a function
taking a `backend` argument. That one argument is the whole difference
between CPU and GPU below.

In [ ]:
function swingup_model(N; T = 5.0, u_max = 4.0, backend = nothing)
    dt = T / N
    c = ExaCore(; backend = backend)
    @add_var(c, θ, 0:N; start = [π * i / N for i = 0:N])
    @add_var(c, ω, 0:N; start = fill(π / T, N + 1))
    @add_var(c, u, 0:N; lvar = -u_max, uvar = u_max, start = zeros(N + 1))

    @add_obj(c, u[i]^2 * dt for i = 0:N)

    @add_con(c, θ[i] - θ[i-1] - dt / 2 * (ω[i] + ω[i-1]) for i = 1:N)
    @add_con(c,
        ω[i] - ω[i-1] - dt / 2 * ((u[i] - 9.81 * sin(θ[i]) - 0.1 * ω[i]) +
                                  (u[i-1] - 9.81 * sin(θ[i-1]) - 0.1 * ω[i-1]))
        for i = 1:N)

    @add_con(c, θ[0])
    @add_con(c, ω[0])
    @add_con(c, θ[N] - π)
    @add_con(c, ω[N])

    return ExaModel(c)
end

Three formulation choices here are deliberate. The boundary conditions are
constraints like any other. ExaModels constraints are equalities unless you
pass `lcon`/`ucon`, and the torque limit went on the *variable* instead.

The initial guess is *dynamically consistent*: θ ramps to π and ω starts at
the matching average rate π/T. This is optional. An interior-point method
does not need a feasible starting point and copes perfectly well with an
inconsistent one, so `start` could be left at zero throughout. Supplying a
guess that roughly satisfies the dynamics simply gives the solver less
distance to travel, which is worth doing when it costs one line.

Build it and solve on the CPU:

In [ ]:
N = 1000
model_cpu = swingup_model(N)
result_cpu = ipopt(model_cpu; max_iter = 600)
result_cpu.status

Each row of the log is one interior-point
iteration. `inf_pr` and `inf_du` are the primal and dual infeasibilities
driving to zero, and `lg(mu)` is the barrier parameter being driven down.
Convergence in a few dozen iterations on a 3000-variable nonconvex problem
is what interior-point methods are good at.

In [ ]:
(objective = result_cpu.objective, iterations = result_cpu.iter)

The solution comes back through `solution`, addressed by the variable
handles our model function returned:

In [ ]:
t_grid = range(0, 5.0; length = N + 1)
θ_sol = Array(solution(result_cpu, model_cpu.refs.θ))
u_sol = Array(solution(result_cpu, model_cpu.refs.u))

Animating the trajectory shows the maneuver directly. The left panel is the
pendulum, drawn from the pivot, with θ = 0 hanging down and θ = π upright.
The right panel is the torque, with a cursor at the current time:

In [ ]:
anim = @animate for k in 1:20:(N + 1)
    pend = plot([0, sin(θ_sol[k])], [0, -cos(θ_sol[k])];
        xlim = (-1.4, 1.4), ylim = (-1.45, 1.35), aspect_ratio = 1,
        lw = 4, marker = :circle, ms = 8, legend = false,
        titlefontsize = 10,
        title = "pendulum at t = $(round(t_grid[k]; digits = 2)) s")
    trace = plot(t_grid, u_sol;
        legend = false, lw = 2, color = :gray, titlefontsize = 10,
        xlabel = "t [s]", title = "torque u(t), |u| ≤ 4")
    vline!(trace, [t_grid[k]]; color = :red, lw = 2)
    plot(pend, trace; layout = (1, 2), size = (820, 380),
        left_margin = 4Plots.mm, bottom_margin = 4Plots.mm, top_margin = 3Plots.mm)
end
gif(anim, fps = 25)

The pumping is visible: the torque swings back and forth to build energy,
then catches the pendulum at the top.

## The device switch

Everything above ran on the CPU. Here is the entire porting effort:

In [ ]:
model_gpu = swingup_model(N; backend = CUDABackend())
result_gpu = madnlp(model_gpu; max_iter = 600, tol = 1e-8)
(status = result_gpu.status, objective = result_gpu.objective,
 cpu_objective = result_cpu.objective)

One argument moved evaluation, derivatives and the KKT factorizations to
the GPU, and the two devices agree.

Note `tol = 1e-8`. MadNLP's GPU default is a loose `1e-4`, which suits the
condensed-space method but not these problems. **Pass `tol = 1e-8` to every
GPU solve.**

---

### (*) Scaling with problem size

> **Optional**
>
> A (*) marks material we may skip in the live session, depending on
> time. Nothing later depends on it, and the cells run on their own.

Grow the horizon and time both devices. Iterations are reported too: an
interior-point method's cost is per iteration, so only runs with similar
counts compare directly. The CPU column is Ipopt and the GPU column is
MadNLP, so the ratio mixes a solver difference into the device difference.

In [ ]:
sizes = [625, 1_250, 2_500, 5_000, 10_000]   # doubling, stopping at 10^4
times = map(sizes) do n
    m_c = swingup_model(n)
    t_c = @elapsed r_c = ipopt(m_c; max_iter = 600, print_level = 0)
    m_g = swingup_model(n; backend = CUDABackend())
    t_g = @elapsed r_g = madnlp(m_g; max_iter = 600, tol = 1e-8,
                                print_level = MadNLP.ERROR)
    (N = n, cpu_s = round(t_c; digits = 2), cpu_iters = r_c.iter,
     gpu_s = round(t_g; digits = 2), gpu_iters = r_g.iter,
     speedup = round(t_c / t_g; digits = 1))
end
times

The same numbers as a curve, with the break-even line marked:

In [ ]:
plot([t.N for t in times], [t.speedup for t in times];
    xscale = :log10, marker = :circle, lw = 2, legend = false,
    xlabel = "N (horizon length)", ylabel = "CPU time / GPU time",
    title = "GPU speedup against problem size")
hline!([1.0]; ls = :dash, color = :black)

Slower or level at small sizes, pulling away as the problem grows: the same
crossover as tutorial 1's batch experiment, now with sparse factorizations
in the mix.

### ✏️ Exercise 1: minimum time

> **Your turn.** The horizon above was fixed at 5 seconds and we minimized
> control effort. Ask the opposite question: with a stronger motor,
> `u_max = 10`, what is the *fastest* swing-up?
>
> The final time has to become part of the problem. Goddard's rocket below
> does exactly this, so look at how it handles a free final time and apply
> the same idea here.
>
> Solve on the GPU, plot θ and u, and compare the torque with the
> minimum-effort solution.

Solution:
[minimum-time-solution](https://madsuite.org/ifac2026/notebooks/minimum-time-solution.html).

---

## Goddard's rocket

Maximize the final altitude of a vertically launched rocket, with bounded
thrust and drag that grows with speed. With altitude $h$, velocity $v$,
mass $m$ and thrust $T$:

![Goddard's rocket: thrust against drag and gravity, burning limited fuel](figs/goddard.svg)

$$
\begin{aligned}
\max_{T,\, t_f}\quad & h(t_f) \\
\text{s.t.}\quad & \dot h = v, \\
& \dot v = \frac{T - D(h, v) - m\, g(h)}{m}, \qquad \dot m = -\frac{T}{c}, \\
& D(h,v) = D_c\, v^2 \exp\!\big(-h_c (h - h_0)/h_0\big), \qquad
  g(h) = g_0 \left(\frac{h_0}{h}\right)^{2}, \\
& h(0) = h_0,\quad v(0) = 0,\quad m(0) = m_0,\quad m(t_f) = m_f, \\
& 0 \le T \le T_{\max}, \qquad h \ge h_0 .
\end{aligned}
$$

Two formulation ideas beyond the swing-up. The final time $t_f$ is *free*,
so the step length becomes a decision variable, and we *maximize* by
passing `minimize = false`.

The third idea is the transcription itself. Everything so far used the
trapezoidal rule, which is second order: accuracy comes only from taking
more steps. Here we use **orthogonal collocation on finite elements**, the
method from Part II of the lecture, which transcribes at high order,
stably for stiff systems, and purely algebraically.

Split the horizon into $N$ elements. On element $i$ of length $h_i$, with a
local coordinate $\tau \in [0,1]$, represent each state as a degree-$K$
Lagrange polynomial through its values at
$\tau_0 = 0 < \tau_1 < \dots < \tau_K$:

$$
z^K(t) = \sum_{j=0}^{K} \ell_j(\tau)\, z_{ij},
\qquad
\ell_j(\tau) = \prod_{k \ne j} \frac{\tau - \tau_k}{\tau_j - \tau_k} .
$$

Since $\ell_j(\tau_k) = \delta_{jk}$, the $z_{ij}$ *are* the state values at
the nodes, and they are the decision variables. We cannot ask
$\dot z^K = f(z^K, u)$ to hold at every $t$ with finitely many unknowns, so
we impose it at the $K$ collocation points, which gives the **collocation
equations**:

$$
\sum_{j=0}^{K} D_{kj}\, z_{ij} \;=\; h_i\, f\big(z_{ik},\, u_{ik}\big),
\qquad i = 1, \dots, N, \quad k = 1, \dots, K,
\qquad D_{kj} = \frac{d\ell_j}{d\tau}(\tau_k) .
$$

Two indices this time, so the pattern repeats $N \times K$ times, and it is
still one generator.

$D$ is a table of precomputed constants; the $z_{ij}$ and $h_i$ are the
unknowns. Each equation is therefore a linear combination of unknowns set
equal to $h_i f(\cdot)$, which is exactly the shape an NLP solver wants.

We use **Radau** points, whose last node sits at $\tau_K = 1$. That choice
gives stiff decay and makes the continuity condition
$z_{i+1,0} = \sum_j \ell_j(1) z_{ij}$ collapse to
$z_{i+1,0} = z_{iK}$, since $\ell_j(1) = \delta_{jK}$: consecutive elements
simply share the node.

$D$ follows from the nodes alone. Differentiating the Lagrange basis is one
small linear solve:

In [ ]:
# Radau IIA nodes for K = 3, giving order 2K-1 = 5 accuracy per element.
const RADAU_C = [0.15505102572168220, 0.64494897427831780, 1.0]

# D[k,j] = dl_j/dtau at tau_k, over the K+1 nodes [0, c...]. Obtained by
# writing the basis in monomials: values at the nodes are V, derivatives at
# the collocation points are Vp, so D = Vp / V.
function differentiation_matrix(c)
    K = length(c)
    τ = vcat(0.0, c)
    V = [τ[i]^(j - 1) for i = 1:K+1, j = 1:K+1]
    Vp = [(j - 1) * c[k]^(max(j - 2, 0)) * (j >= 2) for k = 1:K, j = 1:K+1]
    return Vp / V
end

D_RADAU = differentiation_matrix(RADAU_C)

Worth a sanity check before trusting it: a constant state has zero
derivative at every collocation point, and the state $z = \tau$ has
derivative one.

In [ ]:
(constant = D_RADAU * ones(4), linear = D_RADAU * vcat(0.0, RADAU_C))

Now the model. Note what the decision variables are: the state at each
element start, the state at each collocation point, one thrust per element,
and the element length itself, since the final time is free.

In [ ]:
function goddard_model(nh; K = 3, backend = nothing)
    c = RADAU_C
    D = differentiation_matrix(c)
    h_0, v_0, m_0, g_0 = 1.0, 0.0, 1.0, 1.0
    T_c, h_c, v_c, m_c = 3.5, 500.0, 620.0, 0.6
    cex = 0.5 * sqrt(g_0 * h_0)
    m_f = m_c * m_0
    D_c = 0.5 * v_c * (m_0 / g_0)
    T_max = T_c * m_0 * g_0

    core = ExaCore(; backend = backend, minimize = false)

    @add_var(core, step, 1; start = 1 / nh, lvar = 0.0)     # element length h_i
    # z_ij: state at node j of element i, with j = 0 the element start
    @add_var(core, h, 1:nh, 0:K; start = h_0, lvar = h_0)
    @add_var(core, v, 1:nh, 0:K; start = 0.1, lvar = v_0)
    @add_var(core, m, 1:nh, 0:K; start = m_0, lvar = m_f, uvar = m_0)
    @add_var(core, Th, 1:nh; start = T_max / 2, lvar = 0.0, uvar = T_max)

    @add_obj(core, h[nh, K])

    # One collocation row per (element, collocation point): the right-hand
    # side h_i f(z_ik), then the sum over j of D[k,j] z_ij added to it. Both
    # index rows through the same formula.
    pts = [(n = (i - 1) * K + k, i = i, k = k) for i = 1:nh for k = 1:K]
    trm = [(n = (i - 1) * K + k, i = i, j = j, d = D[k, j+1])
           for i = 1:nh for k = 1:K for j = 0:K]

    @add_con(core, ch, -step[1] * v[e.i, e.k] for e in pts)
    @add_con!(core, ch[e.n] += e.d * h[e.i, e.j] for e in trm)

    @add_con(core, cv, -step[1] *
        ((Th[e.i] - D_c * v[e.i, e.k]^2 * exp(-h_c * (h[e.i, e.k] - h_0) / h_0) -
          m[e.i, e.k] * g_0 * (h_0 / h[e.i, e.k])^2) / m[e.i, e.k]) for e in pts)
    @add_con!(core, cv[e.n] += e.d * v[e.i, e.j] for e in trm)

    @add_con(core, cm, step[1] * Th[e.i] / cex for e in pts)
    @add_con!(core, cm[e.n] += e.d * m[e.i, e.j] for e in trm)

    # continuity: the last node of an element is the first node of the next
    @add_con(core, h[i, K] - h[i+1, 0] for i = 1:nh-1)
    @add_con(core, v[i, K] - v[i+1, 0] for i = 1:nh-1)
    @add_con(core, m[i, K] - m[i+1, 0] for i = 1:nh-1)

    @add_con(core, h[1, 0] - h_0)
    @add_con(core, v[1, 0] - v_0)
    @add_con(core, m[1, 0] - m_0)
    @add_con(core, m[nh, K] - m_f)

    return ExaModel(core)
end

The published optimal final altitude for this problem is **1.01283** in
normalized units (COPS 3.0), which gives us something independent to check
the answer against:

In [ ]:
# 200 intervals here, against the 1000 the trapezoidal swing-up needed:
# higher order buys accuracy that a finer grid would otherwise have to.
nh = 200
rocket = goddard_model(nh; backend = CUDABackend())
rocket_result = madnlp(rocket; max_iter = 600, tol = 1e-8)
(status = rocket_result.status, final_altitude = rocket_result.objective,
 published_optimum = 1.01283)

The altitude matches the literature. Checking optimizer output against
something independent matters: `SOLVE_SUCCEEDED` means the algorithm met
its own criterion, not that you modelled the right problem.
And the trajectories:

In [ ]:
h_traj = Array(solution(rocket_result, rocket.refs.h))[:, 1]   # node j=0 of each element
Th_traj = Array(solution(rocket_result, rocket.refs.Th))  # nh interval controls

p1 = plot(range(0, 1; length = nh), h_traj; lw = 2, legend = false,
    title = "Goddard: altitude")
p2 = plot(range(0, 1; length = nh), Th_traj; lw = 2, legend = false,
    title = "thrust (note the singular arc)", xlabel = "normalized time")
plot(p1, p2; layout = (2, 1), size = (760, 460),
    left_margin = 4Plots.mm, bottom_margin = 4Plots.mm)

The thrust profile has three phases: full burn, then a *singular arc*, a
sustained intermediate thrust balancing drag against gravity, then cutoff.

## Particle steering

The **particle steering** problem (COPS calls it that; you will also see it
as "rocket steering") is a minimum-time problem: steer a constant-magnitude
thrust vector so a particle reaches a given height with a given terminal
velocity as fast as possible. With
position $(x_1, x_2)$, velocity $(x_3, x_4)$, steering angle $u$ and
acceleration magnitude $a = 100$:

$$
\begin{aligned}
\min_{u,\, t_f}\quad & t_f \\
\text{s.t.}\quad & \dot x_1 = x_3, \qquad \dot x_2 = x_4, \\
& \dot x_3 = a \cos u, \qquad \dot x_4 = a \sin u, \\
& x(0) = 0, \qquad x_2(t_f) = 5,\ x_3(t_f) = 45,\ x_4(t_f) = 0, \\
& -\tfrac{\pi}{2} \le u(t) \le \tfrac{\pi}{2} .
\end{aligned}
$$

Free final time again, here as an explicit `tf` variable that *is* the
objective, with states in a 2-D matrix variable, indexed like an array.

A particle starts at rest at the origin. Its engine pushes with a fixed magnitude, and the only thing you
choose is the *direction* of that push at each instant. The target is a
state, not a place: be at height 5, travelling horizontally at 45, with no
vertical motion left. Reach it as quickly as possible.

![Particle steering: thrust direction is the decision](figs/steering.svg)

The arrows are the decision. Point them too steeply and the particle gains
height quickly but arrives with vertical speed it must then cancel; point
them too flat and it never reaches the height. The optimizer finds the
schedule that trades these against each other in the least time.

> **Sources**
>
> Classical problem: A. E. Bryson and Y.-C. Ho, *Applied Optimal
> Control*, Wiley, 1975, pp. 59–62. The thrust magnitude and terminal
> conditions used here follow J. Betts, S. Eldersveld and W. Huffman,
> *Sparse nonlinear programming test problems (Release 1.0)*, Boeing
> Computer Services technical report BCSTECH-93-047, 1993, as collected
> in COPS 3.0 (problem 9): Dolan, Moré and Munson, ANL/MCS-TM-273, 2004.

---

### ✏️ Exercise 2: transcribe it yourself

> **Your turn.** You have the continuous-time statement above and nothing
> else. Transcribe it into an ExaModels model and solve it on the GPU with
> `tol = 1e-8`.
>
> **The discretization is your choice**: implicit Euler, the trapezoidal
> rule, or collocation. They differ in accuracy per grid point and in how
> many variables they cost, so the method you pick changes both the answer
> and the size of the problem.
>
> Report the status, the minimal flight time, and the iteration count. The
> published optimum is 0.55457; COPS reports 0.554577 at `nh` = 200 and
> 0.554571 at `nh` = 800, so the discretized optimum shifts slightly as the
> mesh is refined. Then plot the steering angle `u` over time and interpret
> what the rocket is doing.

In [ ]:
# your code here

Solution:
[particle-steering-solution](https://madsuite.org/ifac2026/notebooks/particle-steering-solution.html).

---

---

## Parameter estimation for dynamic models

> **Optional material**
>
> This section is extra material. We may not work through it live,
> depending on time. It is written to be read and run on your own
> afterwards, and nothing earlier in the workshop depends on it.

The problems above chose a control to reach a target. Estimation inverts
the question: a system of ODEs carries unknown parameters, measurements of
it are noisy, and the task is to find the parameters that best explain the
data.

The machinery is the one you have already used. We discretize the dynamics
into constraints as before, the states on the grid become decision
variables as before, and the only changes are that the unknown parameters
join the variable list and the objective measures misfit against data
instead of control cost. This is the **simultaneous** (or "all-at-once")
approach to dynamic optimization, and because it produces one large
structured NLP, everything the GPU did for us above applies unchanged.

### The system

The JAK2/STAT5 signaling model of Boehm et al. (2014), a standard benchmark
for parameter estimation.

![STAT5 phosphorylation, dimerization, nuclear import and export](figs/stat5.svg)

STAT5 comes in two isoforms. A stimulus that decays over time phosphorylates
them, they pair into three dimers, and those are imported into the nucleus
and later exported back as monomers. Mass spectrometry measures the
phosphorylated forms over four hours. The rate constants
$k_{\text{phos}}$, $k_{\text{imp}}$, $k_{\text{exp}}$ and $k_{\text{deg}}$
are what we estimate.

For our purposes it is an 8-state nonlinear ODE with 9 unknown parameters
and 48 noisy measurements: the shape of estimation problems in process
control and systems biology alike.

> **Sources**
>
> Model and data: M. E. Boehm, L. Adlung, M. Schilling, S. Roth,
> U. Klingmüller and W. D. Lehmann, "Identification of Isoform-Specific
> Dynamics in Phosphorylation-Dependent STAT5 Dimerization by
> Quantitative Mass Spectrometry and Mathematical Modeling", *Journal of
> Proteome Research* 13(12):5685–5694, 2014 (doi:10.1021/pr5006923).
> Distributed as problem `Boehm_JProteomeRes2014` of the PEtab benchmark
> collection (H. Hass et al., "Benchmark problems for dynamic modeling
> of intracellular processes", *Bioinformatics* 35:3073–3082, 2019
> doi:10.1093/bioinformatics/btz020), in the PEtab format of
> L. Schmiester et al., *PLOS Computational Biology* 17(1):e1008646, 2021
> (doi:10.1371/journal.pcbi.1008646). We take the problem from that
> collection and write it out directly, as earlier in this tutorial.

### The dynamics

Eight species: cytosolic `STAT5A`, `STAT5B`, the three phosphorylated
dimers `pApA`, `pApB`, `pBpB`, and their nuclear counterparts `nucpApA`,
`nucpApB`, `nucpBpB`. Nine mass-action reactions, namely phosphorylation-
driven dimerization (3), nuclear import (3) and nuclear export (3), with
cytosolic
and nuclear compartment volumes $V_c = 1.4$, $V_n = 0.45$:

$$
\begin{aligned}
v_1 &= V_c\, E(t)\, k_{\text{phos}}\, \mathrm{STAT5A}^2,
& v_4 &= V_c\, k_{\text{imp,homo}}\, \mathrm{pApA},
& v_7 &= V_n\, k_{\text{exp,homo}}\, \mathrm{nucpApA}, \\
v_2 &= V_c\, E(t)\, k_{\text{phos}}\, \mathrm{STAT5A}\,\mathrm{STAT5B},
& v_5 &= V_c\, k_{\text{imp,hetero}}\, \mathrm{pApB},
& v_8 &= V_n\, k_{\text{exp,hetero}}\, \mathrm{nucpApB}, \\
v_3 &= V_c\, E(t)\, k_{\text{phos}}\, \mathrm{STAT5B}^2,
& v_6 &= V_c\, k_{\text{imp,homo}}\, \mathrm{pBpB},
& v_9 &= V_n\, k_{\text{exp,homo}}\, \mathrm{nucpBpB},
\end{aligned}
$$

where the stimulus decays exponentially, $E(t) = 1.25\times10^{-7}
\exp(-k_{\text{deg}}\, t)$. The balances (note the stoichiometric 2s: two
monomers make one homodimer) are

$$
\begin{aligned}
V_c\, \tfrac{d}{dt}\mathrm{STAT5A} &= -2v_1 - v_2 + 2v_7 + v_8,
& V_c\, \tfrac{d}{dt}\mathrm{pApA} &= v_1 - v_4,
& V_n\, \tfrac{d}{dt}\mathrm{nucpApA} &= v_4 - v_7, \\
V_c\, \tfrac{d}{dt}\mathrm{STAT5B} &= -2v_3 - v_2 + 2v_9 + v_8,
& V_c\, \tfrac{d}{dt}\mathrm{pApB} &= v_2 - v_5,
& V_n\, \tfrac{d}{dt}\mathrm{nucpApB} &= v_5 - v_8, \\
& & V_c\, \tfrac{d}{dt}\mathrm{pBpB} &= v_3 - v_6,
& V_n\, \tfrac{d}{dt}\mathrm{nucpBpB} &= v_6 - v_9 .
\end{aligned}
$$

### What is measured

The instrument does not see the states. It sees three *relative* quantities
the fraction of STAT5A that is phosphorylated, the same for STAT5B, and the
relative amount of STAT5A overall. Each is a nonlinear function of the
states involving a known isoform-specificity constant $c_{17} = 0.107$:

$$
\begin{aligned}
y_1 &= \frac{100\,\mathrm{pApB} + 200\,\mathrm{pApA}\, c_{17}}
             {\mathrm{pApB} + \mathrm{STAT5A}\, c_{17} + 2\,\mathrm{pApA}\, c_{17}}, \\[2pt]
y_2 &= \frac{-100\,\mathrm{pApB} + 200\,\mathrm{pBpB}\,(c_{17}-1)}
             {\mathrm{STAT5B}\,(c_{17}-1) - \mathrm{pApB} + 2\,\mathrm{pBpB}\,(c_{17}-1)}, \\[2pt]
y_3 &= \frac{100\,\mathrm{pApB} + 100\,\mathrm{STAT5A}\, c_{17} + 200\,\mathrm{pApA}\, c_{17}}
             {2\,\mathrm{pApB} + \mathrm{STAT5A}\, c_{17} + 2\,\mathrm{pApA}\, c_{17}
              - \mathrm{STAT5B}\,(c_{17}-1) - 2\,\mathrm{pBpB}\,(c_{17}-1)} .
\end{aligned}
$$

Observation functions like these are the norm rather than the exception,
because you almost never measure the state directly. Note they are
*rational*, so the objective is not a least-squares problem in the standard
sense but a nonconvex one.

The data, 16 time points from 0 to 240 minutes and three observables, ships
next to this notebook as literal arrays:

In [ ]:
include("boehm_data.jl")
(n_timepoints = length(T_DATA), t_final = T_DATA[end], n_measurements = 3 * length(T_DATA))

### Estimation as a nonlinear program

Three choices define it.

**The dynamics become constraints**, by the same orthogonal collocation
used for Goddard. Nuclear import here is fast enough to make the system
stiff, which rules out the trapezoidal rule: it is A-stable but not
L-stable, so it rings on fast modes rather than damping them. Radau
collocation damps them and is high order, so it costs nothing to prefer it.
The elements are 2.5 minutes long, which puts every one of the 16
measurement times on an element boundary.

**The parameters become variables.** The rate constants span nine orders of
magnitude, so we estimate $\theta = \log_{10} k$ and write $k = 10^{\theta}$
in the dynamics. Positivity is then automatic and the search space is well
scaled.

**The objective is a likelihood.** With Gaussian noise of unknown standard
deviation $\sigma_j$ per observable, dropping constants:

$$
\min_{\theta,\, \sigma,\, x(\cdot)} \quad
\sum_{j=1}^{3} \sum_{i=1}^{16}
\left[ \frac{\big(y_j(t_i) - \bar y_{j,i}\big)^2}{2\sigma_j^2}
       + \log \sigma_j \right] .
$$

The $\log \sigma_j$ term stops the optimizer explaining the data by
declaring the noise enormous. We estimate $\log_{10}\sigma$ for the same
reason as the rate constants.

In ExaModels:

In [ ]:
function boehm_model(N; K=3, backend=nothing)
    D = differentiation_matrix(RADAU_C)
    Vc, Vn = 1.4, 0.45; c17, ratio = 0.107, 0.693
    A0, B0 = 207.6*ratio, 207.6*(1-ratio)
    tf = T_DATA[end]; h = tf/N
    elem = [round(Int, t/h) for t in T_DATA]     # element index whose END is t (0 -> start)
    data = [PSTAT5A_REL, PSTAT5B_REL, RSTAT5A_REL]
    c = ExaCore(; backend=backend)
    @add_var(c, θ, 1:6; start=0.0, lvar=-5.0, uvar=5.0)
    @add_var(c, logσ, 1:3; start=0.5, lvar=-3.0, uvar=3.0)
    # z_ij: state at node j of element i; 8 species
    x0 = zeros(N, K+1, 8); x0[:,:,1] .= A0; x0[:,:,2] .= B0
    @add_var(c, x, 1:N, 0:K, 1:8; start=x0, lvar=0.0)
    # observables at a measurement time: node K of that element (node 0 of element 1 for t=0)
    idx = [(e == 0 ? (1,0) : (e,K)) for e in elem]
    yA(i,j) = (100*x[i,j,4] + 200*x[i,j,3]*c17) / (x[i,j,4] + x[i,j,1]*c17 + 2*x[i,j,3]*c17)
    yB(i,j) = (-100*x[i,j,4] + 200*x[i,j,5]*(c17-1)) / (x[i,j,2]*(c17-1) - x[i,j,4] + 2*x[i,j,5]*(c17-1))
    yR(i,j) = (100*x[i,j,4] + 100*x[i,j,1]*c17 + 200*x[i,j,3]*c17) /
              (2*x[i,j,4] + x[i,j,1]*c17 + 2*x[i,j,3]*c17 - x[i,j,2]*(c17-1) - 2*x[i,j,5]*(c17-1))
    obsA = [(i=p[1], j=p[2], d=d) for (p,d) in zip(idx, data[1])]
    obsB = [(i=p[1], j=p[2], d=d) for (p,d) in zip(idx, data[2])]
    obsR = [(i=p[1], j=p[2], d=d) for (p,d) in zip(idx, data[3])]
    @add_obj(c, (yA(e.i,e.j)-e.d)^2/(2*(10.0^logσ[1])^2) for e in obsA)
    @add_obj(c, (yB(e.i,e.j)-e.d)^2/(2*(10.0^logσ[2])^2) for e in obsB)
    @add_obj(c, (yR(e.i,e.j)-e.d)^2/(2*(10.0^logσ[3])^2) for e in obsR)
    @add_obj(c, length(T_DATA)*log(10.0)*logσ[q] for q=1:3)
    # dynamics at each collocation point
    # the time of each collocation point is data, not a symbolic index
    E(e) = 1.25e-7*exp(-(10.0^θ[1])*e.t)
    v1(e)=Vc*E(e)*(10.0^θ[2])*x[e.i,e.k,1]^2; v2(e)=Vc*E(e)*(10.0^θ[2])*x[e.i,e.k,1]*x[e.i,e.k,2]
    v3(e)=Vc*E(e)*(10.0^θ[2])*x[e.i,e.k,2]^2; v4(e)=Vc*(10.0^θ[3])*x[e.i,e.k,3]
    v5(e)=Vc*(10.0^θ[4])*x[e.i,e.k,4]; v6(e)=Vc*(10.0^θ[3])*x[e.i,e.k,5]
    v7(e)=Vn*(10.0^θ[5])*x[e.i,e.k,6]; v8(e)=Vn*(10.0^θ[6])*x[e.i,e.k,7]; v9(e)=Vn*(10.0^θ[5])*x[e.i,e.k,8]
    f = (e->(-2v1(e)-v2(e)+2v7(e)+v8(e))/Vc, e->(-2v3(e)-v2(e)+2v9(e)+v8(e))/Vc,
         e->(v1(e)-v4(e))/Vc, e->(v2(e)-v5(e))/Vc, e->(v3(e)-v6(e))/Vc,
         e->(v4(e)-v7(e))/Vn, e->(v5(e)-v8(e))/Vn, e->(v6(e)-v9(e))/Vn)
    pts = [(n=(i-1)*K+k, i=i, k=k, t=(i-1)*h + RADAU_C[k]*h) for i=1:N for k=1:K]
    trm = [(n=(i-1)*K+k, i=i, j=j, d=D[k,j+1]) for i=1:N for k=1:K for j=0:K]
    @add_con(c, cc1, -h*f[1](e) for e in pts)
    @add_con!(c, cc1[e.n] += e.d*x[e.i,e.j,1] for e in trm)
    @add_con(c, cc2, -h*f[2](e) for e in pts)
    @add_con!(c, cc2[e.n] += e.d*x[e.i,e.j,2] for e in trm)
    @add_con(c, cc3, -h*f[3](e) for e in pts)
    @add_con!(c, cc3[e.n] += e.d*x[e.i,e.j,3] for e in trm)
    @add_con(c, cc4, -h*f[4](e) for e in pts)
    @add_con!(c, cc4[e.n] += e.d*x[e.i,e.j,4] for e in trm)
    @add_con(c, cc5, -h*f[5](e) for e in pts)
    @add_con!(c, cc5[e.n] += e.d*x[e.i,e.j,5] for e in trm)
    @add_con(c, cc6, -h*f[6](e) for e in pts)
    @add_con!(c, cc6[e.n] += e.d*x[e.i,e.j,6] for e in trm)
    @add_con(c, cc7, -h*f[7](e) for e in pts)
    @add_con!(c, cc7[e.n] += e.d*x[e.i,e.j,7] for e in trm)
    @add_con(c, cc8, -h*f[8](e) for e in pts)
    @add_con!(c, cc8[e.n] += e.d*x[e.i,e.j,8] for e in trm)
    @add_con(c, x[i,K,s] - x[i+1,0,s] for i=1:N-1, s=1:8)
    @add_con(c, x[1,0,1] - A0); @add_con(c, x[1,0,2] - B0)
    @add_con(c, x[1,0,s] for s=3:8)
    return ExaModel(c)
end

Solve it on the GPU, with one departure from the rule above: `tol = 1e-6`
rather than `1e-8`. This problem is the exception. At `1e-8` it converged
in one run out of six we tried, the others wandering for thousands of
iterations toward a worse local optimum; at `1e-6` it converged in all
six, in 49 to 193 iterations.

In [ ]:
N = 96           # 96 elements of 2.5 min: every measurement time is a boundary
model = boehm_model(N; backend = CUDABackend())
result = madnlp(model; max_iter = 600, tol = 1e-6)
(status = result.status, objective = result.objective, iterations = result.iter)

### The fitted parameters

Back on the natural scale, since these are $10^{\theta}$:

In [ ]:
θ̂ = Array(solution(result, model.refs.θ))
names = ["k_deg", "k_phos", "k_imp_homo", "k_imp_hetero", "k_exp_homo", "k_exp_hetero"]
[(names[i], 10.0^θ̂[i]) for i = 1:6]

### The fit

The basic check on any estimation is to overlay the fitted model's
predictions on the data it was fitted to:

In [ ]:
# x̂ is indexed (element, node, species); the element boundaries are node 0
# of the first element followed by node K of each one.
x̂ = Array(solution(result, model.refs.x))
K = 3
c17 = 0.107
bnd = vcat([x̂[1, 1, :]], [x̂[i, K + 1, :] for i = 1:N])

yA = [(100 * z[4] + 200 * z[3] * c17) /
      (z[4] + z[1] * c17 + 2 * z[3] * c17) for z in bnd]
yB = [(-100 * z[4] + 200 * z[5] * (c17 - 1)) /
      (z[2] * (c17 - 1) - z[4] + 2 * z[5] * (c17 - 1)) for z in bnd]
t_grid = range(0, T_DATA[end]; length = N + 1)

plt = plot(t_grid, [yA yB]; lw = 2, label = ["pSTAT5A_rel (fit)" "pSTAT5B_rel (fit)"],
    xlabel = "time [min]", ylabel = "relative phosphorylation [%]",
    title = "Boehm STAT5 model: fit vs. data")
scatter!(plt, T_DATA, PSTAT5A_REL; label = "pSTAT5A_rel (data)", ms = 5)
scatter!(plt, T_DATA, PSTAT5B_REL; label = "pSTAT5B_rel (data)", ms = 5)

---

*This notebook was generated using [Literate.jl](https://github.com/fredrikekre/Literate.jl).*